# 주제 ③ 프레스 유압펌프 진동·전류 시계열 — 추가 진단 (02_diagnosis_deep)

- 목적: `01_data_quality.ipynb`의 주장 중 근거가 약했던 항목을 검증하고, 모델 단계에서 바로 쓸 세그먼트 피처표·등급·운전 상태를 코드화한다.
- 모델링 경쟁이 아니라 **데이터 진단**이다. 각 절은 "가설 → 실험 → 결과 → 모델 단계 반영" 순으로 적는다.
- 근거가 간접적인 판단은 **(추정)** 으로 표기한다.
- **v2 개정**: 1차 검증에서 지적된 결함(DC 오프셋 AUC 계산 버그, 표본크기 오류 추론, 전류 채널의 파일 형식 누수, 시간추세 결론 오류, 윈도우 비교의 선택 효과, 세그먼트 단위 오경보율 누락 등)을 반영해 전면 개정했다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
import data_quality as dq
import segments as sg
import signal_checks as sc
FIG, RES = paths.nb_dirs("02_diagnosis_deep")
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
def save(name): plt.tight_layout(); plt.savefig(FIG / name, dpi=110); plt.close(); print("saved", name)

dfs = dq.load_all()
N, O = dfs["normal"], dfs["outlier"]
print(f"normal {N.shape}, outlier {O.shape}, normal seg {N.seg.nunique()}, outlier seg {O.seg.nunique()}")

normal (20000, 9), outlier (600, 9), normal seg 599, outlier seg 21


## 1. 세그먼트 등급화·운전 상태 (B-2 ①)

- **가설**: 이상 21세그먼트는 진동 세기로 3등급(확실 이상/경계/정상 유사)이 나뉘고, 정상 599세그먼트는 부하가 다른 여러 운전 상태로 나뉜다. **(v2 추가)** 진동 기준 등급과 전류 기준 등급이 항상 같지는 않을 수 있다 — 채널마다 별도로 매겨 비교한다.
- **실험**: `segments.segment_features`로 세그먼트별 진동 AI0/AI1 RMS·피크·첨도·crest factor·DC 오프셋(및 절대값), 전류 RMS·피크·DC 오프셋(및 절대값)을 만든다. 이상 세그먼트는 정상 세그먼트의 `vib_rms`(=√(AI0_rms²+AI1_rms²)) 분위수로 진동 등급을, `AI2_Current_dc_abs`(전류 DC 오프셋 절대값) 분위수로 전류 등급을 **각각** 매긴다(둘 다 정상 q50/q99 기준). 정상 세그먼트는 표준화된 피처에 k-means(k=2~4)를 적용해 실루엣 점수로 k를 고른다.

In [2]:
fN = sg.segment_features(N)
fO = sg.segment_features(O)
print("정상 세그먼트 피처표:", fN.shape, " 이상 세그먼트 피처표:", fO.shape)
display(fN.head(2))

정상 세그먼트 피처표: (599, 21)  이상 세그먼트 피처표: (21, 21)


,src,start,len,label,AI0_Vibration_rms,AI0_Vibration_peak,AI0_Vibration_kurt,AI0_Vibration_crest,AI0_Vibration_dc,AI1_Vibration_rms,AI1_Vibration_peak,AI1_Vibration_kurt,AI1_Vibration_crest,AI1_Vibration_dc,AI2_Current_rms,AI2_Current_peak,AI2_Current_dc,vib_rms,AI0_Vibration_dc_abs,AI1_Vibration_dc_abs,AI2_Current_dc_abs
seg_id,,,,,,,,,,,,,,,,,,,,,
0,normal,2022-07-12 00:00:00.019,41,0.0,0.069009,0.191580,0.169165,2.776144,-0.006957,0.163178,0.275627,-1.341630,1.689117,-0.009436,149.170481,219.93151,19.085886,0.177171,0.006957,0.009436,19.085886
1,normal,2022-07-12 00:00:07.224,37,0.0,0.073404,0.136729,-0.938016,1.862683,0.007061,0.132412,0.244895,-1.310623,1.849498,0.006813,139.745235,217.53814,-5.676624,0.151397,0.007061,0.006813,5.676624


In [3]:
fO["vib_grade"] = sg.grade_outlier_segments(fO, fN, col="vib_rms")
fO["cur_grade"] = sg.grade_outlier_segments(fO, fN, col="AI2_Current_dc_abs")
q50, q99 = fN["vib_rms"].quantile([.5, .99])
cq50, cq99 = fN["AI2_Current_dc_abs"].quantile([.5, .99])
print(f"정상 vib_rms 분위수: q50={q50:.4f}, q99={q99:.4f}")
print(f"정상 |AI2_Current_dc| 분위수: q50={cq50:.4f}, q99={cq99:.4f}")
display(fO[["len", "vib_rms", "vib_grade", "AI2_Current_dc", "AI2_Current_dc_abs", "cur_grade"]])
print("진동 등급:", fO["vib_grade"].value_counts().to_dict())
print("전류 등급:", fO["cur_grade"].value_counts().to_dict())
mismatch = fO[fO["vib_grade"] != fO["cur_grade"]]
print(f"\n두 등급이 다른 세그먼트 {len(mismatch)}개 (전류 채널이 진동과 별개의 이상 신호를 담고 있을 수 있음):")
display(mismatch[["len", "vib_grade", "cur_grade", "AI2_Current_dc"]])
print("\n특히 세그먼트 19·20은 진동=정상 유사이지만 전류=확실 이상 — '진동은 조용하지만 전류 DC가 거의 직류 수준으로 치우친' 세그먼트다(3절에서 원인을 다룬다).")

정상 vib_rms 분위수: q50=0.1157, q99=0.2412
정상 |AI2_Current_dc| 분위수: q50=4.8601, q99=154.1167


,len,vib_rms,vib_grade,AI2_Current_dc,AI2_Current_dc_abs,cur_grade
seg_id,,,,,,
0,4,0.863062,확실 이상,-24.735931,24.735931,경계
1,50,0.465965,확실 이상,43.988233,43.988233,경계
2,47,0.591424,확실 이상,-9.257744,9.257744,경계
3,31,0.111847,정상 유사,73.448313,73.448313,경계
4,18,0.760067,확실 이상,-120.268942,120.268942,경계
5,3,1.024817,확실 이상,-141.859070,141.859070,경계
6,50,0.614944,확실 이상,-74.386605,74.386605,경계
7,40,0.408483,확실 이상,-81.419954,81.419954,경계
8,25,0.421369,확실 이상,-26.226046,26.226046,경계


진동 등급: {'확실 이상': 18, '정상 유사': 3}
전류 등급: {'경계': 13, '확실 이상': 7, '정상 유사': 1}

두 등급이 다른 세그먼트 16개 (전류 채널이 진동과 별개의 이상 신호를 담고 있을 수 있음):


,len,vib_grade,cur_grade,AI2_Current_dc
seg_id,,,,
0,4,확실 이상,경계,-24.735931
1,50,확실 이상,경계,43.988233
2,47,확실 이상,경계,-9.257744
3,31,정상 유사,경계,73.448313
4,18,확실 이상,경계,-120.268942
5,3,확실 이상,경계,-141.859070
6,50,확실 이상,경계,-74.386605
7,40,확실 이상,경계,-81.419954
8,25,확실 이상,경계,-26.226046



특히 세그먼트 19·20은 진동=정상 유사이지만 전류=확실 이상 — '진동은 조용하지만 전류 DC가 거의 직류 수준으로 치우친' 세그먼트다(3절에서 원인을 다룬다).


In [4]:
best_k, results, X = sg.choose_k_operating_states(fN)
for k, (labels, sil) in results.items():
    print(f"k={k}: silhouette={sil:.4f}, sizes={np.bincount(labels).tolist()}")
print(f"선택된 k = {best_k}")
fN["state"] = results[best_k][0]
summ = sg.operating_state_summary(fN)
display(summ)

k=2: silhouette=0.5991, sizes=[252, 347]
k=3: silhouette=0.4930, sizes=[279, 142, 178]
k=4: silhouette=0.4374, sizes=[148, 197, 137, 117]
선택된 k = 2


,AI0_Vibration_rms,AI1_Vibration_rms,AI2_Current_rms,AI2_Current_peak,n_segments,n_samples,share_samples
state,,,,,,,
0,0.084,0.164,156.344,229.196,252,8635,0.4318
1,0.055,0.060,86.446,120.705,347,11365,0.5682


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = {"확실 이상": "crimson", "경계": "orange", "정상 유사": "steelblue"}
order = fO.sort_values("start").index
ax.bar(range(len(order)), fO.loc[order, "vib_rms"], color=[colors[g] for g in fO.loc[order, "vib_grade"]])
ax.axhline(q50, ls="--", c="steelblue", lw=1, label=f"정상 q50={q50:.3f}")
ax.axhline(q99, ls="--", c="crimson", lw=1, label=f"정상 q99={q99:.3f}")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=0)
ax.set_xlabel("이상 세그먼트 번호(시간순)"); ax.set_ylabel("vib_rms"); ax.set_title("이상 21세그먼트 진동 기준 등급 (정상 vib_rms 분위수 기준)")
ax.legend(fontsize=8)
save("02_segment_grades.png")

saved 02_segment_grades.png


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for s, marker in zip(sorted(fN["state"].unique()), ["o", "s", "^", "D"]):
    sub = fN[fN["state"] == s]
    axes[0].scatter(sub["AI2_Current_rms"], sub["vib_rms"], s=10, alpha=.5, label=f"state {s}", marker=marker)
axes[0].set_xlabel("AI2_Current_rms"); axes[0].set_ylabel("vib_rms"); axes[0].set_title("정상 운전 상태(k-means) — 전류 RMS vs 진동 RMS"); axes[0].legend(fontsize=8)
fN.sort_values("start")["state"].reset_index(drop=True).plot(ax=axes[1], lw=.6)
axes[1].set_title("세그먼트 순서(시간)에 따른 운전 상태"); axes[1].set_xlabel("세그먼트 순번"); axes[1].set_ylabel("state")
save("02_operating_states.png")

saved 02_operating_states.png


In [7]:
# 01 리포트가 지목한 "60~68분 저부하 구간"(행 순서 10분위 중 8번째 청크)이 실제로 어떤 state인지 확인
N2 = N.copy()
N2["chunk"] = pd.qcut(np.arange(len(N2)), 10, labels=False)
chunk8 = N2[N2["chunk"] == 8]
t0, t1 = (chunk8["ts"].min() - N2["ts"].min()).total_seconds() / 60, (chunk8["ts"].max() - N2["ts"].min()).total_seconds() / 60
print(f"청크8 시간 범위: {t0:.1f}~{t1:.1f}분, 전류 |값| 최댓값={chunk8['AI2_Current'].abs().max():.1f} (01 리포트의 '60~68분, 전류 최대 118'과 일치)")
segs_in_chunk8 = chunk8["seg"].unique()
states_in_chunk8 = fN.loc[fN.index.isin(segs_in_chunk8), "state"]
print(f"이 구간에 걸친 세그먼트 {len(segs_in_chunk8)}개 중 state 분포: {states_in_chunk8.value_counts().to_dict()}")

청크8 시간 범위: 61.0~68.8분, 전류 |값| 최댓값=118.2 (01 리포트의 '60~68분, 전류 최대 118'과 일치)
이 구간에 걸친 세그먼트 59개 중 state 분포: {1: 59}


**1절 결과**: (수치는 위 출력에서 인용) 진동 등급과 전류 등급이 다른 세그먼트가 존재한다 — 특히 19·20은 진동으로는 "정상 유사"이지만 전류로는 "확실 이상"이다. 01 리포트가 지목한 60~68분 저부하 구간은 k-means state와 정확히 일치한다.

**모델 단계 반영**: `data/processed/segments.csv`(아래 셀에서 생성)를 GroupKFold의 그룹 키·soft label 원천으로 쓴다. 등급은 **채널별로 분리해서** soft label 후보로 쓴다(`vib_grade`, `cur_grade`) — 세그먼트 19·20처럼 한쪽 채널만 이상을 보이는 경우를 "정상"으로 단정하면 안 된다(2·3절에서 원인 규명). `state`는 정상 세그먼트의 운전 상태 피처(또는 층화 기준)로 쓴다.

In [8]:
seg_table = sg.build_segments_table()
print("data/processed/segments.csv 저장:", seg_table.shape)
display(seg_table[seg_table.src == "outlier"][["seg_uid", "len", "vib_grade", "cur_grade"]])

data/processed/segments.csv 저장: (620, 26)


,seg_uid,len,vib_grade,cur_grade
599,outlier_0,4,확실 이상,경계
600,outlier_1,50,확실 이상,경계
601,outlier_2,47,확실 이상,경계
602,outlier_3,31,정상 유사,경계
603,outlier_4,18,확실 이상,경계
604,outlier_5,3,확실 이상,경계
605,outlier_6,50,확실 이상,경계
606,outlier_7,40,확실 이상,경계
607,outlier_8,25,확실 이상,경계
608,outlier_9,8,확실 이상,경계


## 2. 신호 해상도·포화·DC 오프셋/파일 형식 누수 (B-2 ②)

- **가설**: (a) 채널값의 최소 간격으로 ADC 비트 수를 추정할 수 있다. (b) 전류 |값|이 특정 상한 근방에서 반복되면 클리핑(포화)이다. (c) 세그먼트 DC 오프셋(평균)만으로 normal/outlier가 갈리면 센서 재장착 등 누수 위험이다. **(v2 추가)** (d) outlier 파일의 낮은 고유값 비율이 표본 크기 때문이 아니라 실제 값 형식(양자화·소수 자릿수)이 다르기 때문일 수 있다 — 표본이 작을수록 연속값의 고유값 비율은 오히려 **높아지므로**(충돌 확률 감소), 낮게 나온다면 이는 표본 크기 효과로 설명할 수 없다.
- **실험**: `signal_checks.resolution_table`(고유값 수·최소 간격·유일값 비율), `clipping_check`(|값| 최댓값 근방 반복), `dc_offset_auc`(**절대값** DC 오프셋 단독 AUC + 진폭 대비 |DC|/RMS AUC — 아래 참고), `quantization_multiple_share`(전류값이 특정 간격의 정수배인 비율), `decimal_places_table`(소수 자릿수 분포).

> **버그 수정**: 1차 버전의 `dc_offset_auc`는 부호 있는 DC 값에 `max(auc,1-auc)`를 적용했는데, 이는 +방향으로 치우친 세그먼트와 −방향으로 치우친 세그먼트가 서로 반대 순위로 기여해 상쇄되는 결함이었다. 이번에는 **|DC|(절대값)** 로 다시 계산한다.

In [9]:
display(sc.resolution_table(dfs))

,src,channel,n_rows,n_unique,unique_ratio,min_step,range,bits_est
0,normal,AI0_Vibration,20000,19124,0.9562,1.000000e-06,0.6663,19.35
1,normal,AI1_Vibration,20000,19460,0.9730,1.000000e-06,0.7663,19.55
2,normal,AI2_Current,20000,19934,0.9967,1.100000e-04,544.8033,22.24
3,outlier,AI0_Vibration,600,594,0.9900,1.103800e-05,3.3059,18.19
4,outlier,AI1_Vibration,600,597,0.9950,1.157000e-05,1.2606,16.73
5,outlier,AI2_Current,600,340,0.5667,1.192090e+00,938.1772,9.62


In [10]:
display(sc.clipping_check(dfs, channel="AI2_Current"))
print("주의: outlier 파일의 고유값 비율이 낮은 것은 아래 셀에서 보듯 실제 양자화 형식 차이 때문이며, 표본 크기(600 vs 20,000) 효과로는 설명되지 않는다 — 표본이 작을수록 연속값의 고유값 비율은 통계적으로 오히려 높아지기 때문이다.")

,src,channel,max_abs,count_at_max,count_ge_99pct_of_max,share_ge_near_max
0,normal,AI2_Current,273.235,1,10,0.0005
1,outlier,AI2_Current,538.826,1,1,0.0017


주의: outlier 파일의 고유값 비율이 낮은 것은 아래 셀에서 보듯 실제 양자화 형식 차이 때문이며, 표본 크기(600 vs 20,000) 효과로는 설명되지 않는다 — 표본이 작을수록 연속값의 고유값 비율은 통계적으로 오히려 높아지기 때문이다.


In [11]:
dc_auc = sc.dc_offset_auc(fN, fO)
display(dc_auc)
print("|DC| AUC가 AI0 0.94·전류 0.89 수준으로 매우 높다 — 세그먼트 평균(DC 오프셋)만으로도 normal/outlier가 거의 갈린다는 뜻. 1차 버전(부호 있는 값+max(auc,1-auc))은 이 신호를 상쇄시켜 0.5 근처로 낮게 오판했었다.")
print()
print("정상(길이>=50) 전류 DC 범위:", fN.loc[fN.len>=50, "AI2_Current_dc"].min(), "~", fN.loc[fN.len>=50, "AI2_Current_dc"].max())
print("이상 전류 DC 범위:", fO["AI2_Current_dc"].min(), "~", fO["AI2_Current_dc"].max())
print("이상 세그먼트 16~20의 전류 DC(직류에 가까움):")
display(fO.loc[16:20, ["len", "AI2_Current_dc", "AI2_Current_rms"]])

,channel,abs_dc_auc,abs_dc_over_rms_auc,leakage_suspect
0,AI0_Vibration,0.9443,0.7870,True
1,AI1_Vibration,0.9198,0.8412,True
2,AI2_Current,0.8911,0.8862,True


|DC| AUC가 AI0 0.94·전류 0.89 수준으로 매우 높다 — 세그먼트 평균(DC 오프셋)만으로도 normal/outlier가 거의 갈린다는 뜻. 1차 버전(부호 있는 값+max(auc,1-auc))은 이 신호를 상쇄시켜 0.5 근처로 낮게 오판했었다.

정상(길이>=50) 전류 DC 범위: -0.6730367599999996 ~ 1.4130353999999983
이상 전류 DC 범위: -184.77441725 ~ 272.4263723333334
이상 세그먼트 16~20의 전류 DC(직류에 가까움):


,len,AI2_Current_dc,AI2_Current_rms
seg_id,,,
16,50,231.885937,256.755919
17,36,272.426372,296.897985
18,24,225.553936,242.081611
19,10,198.483489,207.236039
20,15,180.085522,183.383022


In [12]:
q_table = sc.quantization_multiple_share(dfs, channel="AI2_Current", step=1.19209, tol=1e-3)
display(q_table)
d_table = sc.decimal_places_table(dfs)
display(d_table)
print("전류값이 1.19209의 정수배(오차 0.1% 이내)인 비율: normal 0.19% vs outlier 99.67% — 두 파일이 서로 다른 계측/전처리 파이프라인을 거쳤다는 강한 증거.")
print("진동값 소수 자릿수도 normal은 6자리가 최빈(약 90%), outlier는 8자리가 최빈(약 44~66%) — 물리적 신호 차이가 아니라 '파일을 어떻게 내보냈는가'라는 형식 차이만으로도 라벨을 맞출 수 있다(파일 형식 누수).")

,src,channel,step,tol,share_multiple_of_step
0,normal,AI2_Current,1.19209,0.001,0.0019
1,outlier,AI2_Current,1.19209,0.001,0.9967


,src,channel,mode_decimals,share_at_mode,decimal_range
0,normal,AI0_Vibration,6,0.8979,2~16
1,normal,AI1_Vibration,6,0.8957,1~16
2,outlier,AI0_Vibration,8,0.4400,5~11
3,outlier,AI1_Vibration,8,0.6567,6~11


전류값이 1.19209의 정수배(오차 0.1% 이내)인 비율: normal 0.19% vs outlier 99.67% — 두 파일이 서로 다른 계측/전처리 파이프라인을 거쳤다는 강한 증거.
진동값 소수 자릿수도 normal은 6자리가 최빈(약 90%), outlier는 8자리가 최빈(약 44~66%) — 물리적 신호 차이가 아니라 '파일을 어떻게 내보냈는가'라는 형식 차이만으로도 라벨을 맞출 수 있다(파일 형식 누수).


**2절 결과(v2, 결론 반전)**: 1차 리포트는 "DC 오프셋 단독 AUC가 낮아 누수 근거 약함"이라고 결론지었으나 **이는 계산 버그였다.** 올바르게(절대값으로) 계산하면 AI0 |DC| AUC **0.9443**, AI2 |DC| AUC **0.8911**로 세그먼트 평균만으로도 파일을 거의 구분할 수 있다. 게다가 전류값의 **99.67%가 1.19209의 정수배**(normal은 0.19%)이고, 진동값의 소수 자릿수 포맷도 파일마다 다르다(normal 6자리 vs outlier 8자리 최빈) — 즉 **normal과 outlier는 서로 다른 계측/저장 파이프라인을 거친 것으로 보이며, 이 형식 차이 자체가 라벨과 100% 대응하는 누수 채널이다.** 이상 세그먼트 16~20의 전류 DC는 180~272로 정상(길이≥50 기준 −0.67~1.41)과 비교할 수 없을 정도로 크며 사실상 직류(DC)에 가깝다.
이 데이터로는 **"설비가 실제로 이상 상태라 전류 DC가 치솟았다"**는 가설과 **"이상 이벤트를 다른 날 별도로 내보내면서 계측 체인/스케일링 설정이 달라졌다"**는 가설을 구분할 수 없다(추정, 한계로 명시).

**모델 단계 반영(v2, 반전)**: DC 오프셋은 더 이상 "선택적 강건성 옵션"이 아니라 **세그먼트 평균 제거를 필수 전처리로 채택**한다(그렇지 않으면 모델이 실제 이상 패턴이 아니라 파일 형식/오프셋 차이를 학습해 버릴 위험이 크다). 다만 오프셋 제거 후에도 `cur_grade`(2절의 |DC| 크기)는 "이 세그먼트가 원래 얼마나 큰 오프셋을 가졌는가"를 나타내는 별도 피처로 남겨, 오프셋 자체가 설비 신호일 가능성도 완전히 버리지 않는다. **원시값의 반올림 등 형식 통일**(소수 자릿수·양자화 스케일을 두 파일에 동일하게 맞추는 전처리)도 전처리 체크리스트에 추가한다.

## 3. 에일리어싱 검증 (B-2 ③)

- **가설**: 전류는 60 Hz AC가 10 Hz 샘플링으로 접힌(에일리어싱) 단일 저주파 성분만 가지고, 진동 채널에는 전류와 같은 주기의 전원 노이즈가 섞이지 않는다. 이상 구간 일부 세그먼트에서 zero-crossing이 사라지는 것은 AC 성분 자체가 흐트러져서가 아니라 **2절에서 확인한 큰 DC 편이가 부호 변화를 가려서**일 수 있다.
- **실험**: (a) 세그먼트별 zero-crossing 주기의 평균·분산(`signal_checks.zero_crossing_by_segment`)과, DC를 제거한 뒤 다시 계산한 zero-crossing(`zero_crossing_demeaned_by_segment`) 비교. (b) 55~65 Hz 대역을 훑어 관측된 에일리어싱 주파수에 가장 잘 맞는 실제 주파수 후보 계산(`alias_candidates`, 0.1Hz·0.01Hz 두 해상도). (c) 대표 정상 세그먼트(50샘플) FFT 확인 후, 길이 50인 **정상 202세그먼트 전부**로 일반화. (d) 진동 채널 자기상관·zero-crossing과 전류 채널의 상관.

In [13]:
zc_cur = sc.zero_crossing_by_segment(N, "AI2_Current", min_len=10).replace([np.inf, -np.inf], np.nan).dropna()
print(f"normal 전류 zero-crossing 주기(샘플): mean={zc_cur.mean():.3f}, sd={zc_cur.std():.3f}, n_seg={len(zc_cur)}")
print(f"  → 초 단위 mean={zc_cur.mean()/10:.3f}s, 겉보기 주파수={10/zc_cur.mean():.4f} Hz")

zc_cur_o_raw = sc.zero_crossing_by_segment(O, "AI2_Current", min_len=10)
n_inf = zc_cur_o_raw.replace([np.inf, -np.inf], np.nan).isna().sum()
zc_cur_o = zc_cur_o_raw.replace([np.inf, -np.inf], np.nan).dropna()
print(f"outlier 전류 zero-crossing 주기(샘플, 원값): mean={zc_cur_o.mean():.3f}, sd={zc_cur_o.std():.3f}, 유효 {len(zc_cur_o)}/{len(zc_cur_o_raw)}, 부호변화 0회(주기 계산 불가) {n_inf}개")

zc_cur_o_dm = sc.zero_crossing_demeaned_by_segment(O, "AI2_Current", min_len=10)
print("\n세그먼트 평균(DC) 제거 후 zero-crossing 주기(원래 inf였던 세그먼트만):")
inf_segs = zc_cur_o_raw.replace([np.inf, -np.inf], np.nan).isna()
display(pd.DataFrame({"dc": O.groupby("seg")["AI2_Current"].mean().loc[inf_segs[inf_segs].index],
                       "zc_raw": zc_cur_o_raw.loc[inf_segs[inf_segs].index],
                       "zc_demeaned": zc_cur_o_dm.loc[inf_segs[inf_segs].index]}))
print("→ 큰 DC 편이 때문에 부호가 한쪽으로 고정돼 zero-crossing이 0이었을 뿐, 평균을 빼면 교차가 다시 나타난다. '전류의 AC 성분 자체가 흐트러졌다'는 1차 해석은 부정확했다.")

normal 전류 zero-crossing 주기(샘플): mean=17.582, sd=2.723, n_seg=530


  → 초 단위 mean=1.758s, 겉보기 주파수=0.5687 Hz
outlier 전류 zero-crossing 주기(샘플, 원값): mean=10.783, sd=7.444, 유효 12/17, 부호변화 0회(주기 계산 불가) 5개

세그먼트 평균(DC) 제거 후 zero-crossing 주기(원래 inf였던 세그먼트만):


,dc,zc_raw,zc_demeaned
16,231.885937,inf,5.555556
17,272.426372,inf,72.000000
18,225.553936,inf,12.000000
19,198.483489,inf,6.666667
20,180.085522,inf,3.333333


→ 큰 DC 편이 때문에 부호가 한쪽으로 고정돼 zero-crossing이 0이었을 뿐, 평균을 빼면 교차가 다시 나타난다. '전류의 AC 성분 자체가 흐트러졌다'는 1차 해석은 부정확했다.


In [14]:
segN = dq.segment_table(N)
long_segs = segN[segN.n == 50].index.tolist()
rep_seg = long_segs[0]
g = N[N.seg == rep_seg]
freqs, amp = sc.fft_spectrum(g["AI2_Current"].to_numpy(), fs=10.0)
peak_idx = np.argsort(amp[1:])[::-1][:5] + 1
peak_freq = freqs[peak_idx[0]]
print("대표 세그먼트(정상, 50샘플) FFT 상위 5개 성분:")
for i in peak_idx:
    print(f"  freq={freqs[i]:.4f} Hz  amp={amp[i]:.2f}")
print(f"\n최댓값 성분 freq={peak_freq:.4f} Hz, 2위 대비 진폭비={amp[peak_idx[0]]/amp[peak_idx[1]]:.1f}배 → 사실상 단일 성분")

# (v2 추가) 길이 50인 정상 202세그먼트 전부로 일반화
ratios, peak_freqs_all = [], []
for seg_id in long_segs:
    gg = N[N.seg == seg_id]
    fq, am = sc.fft_spectrum(gg["AI2_Current"].to_numpy(), fs=10.0)
    idx2 = np.argsort(am[1:])[::-1][:2] + 1
    peak_freqs_all.append(fq[idx2[0]])
    ratios.append(am[idx2[0]] / am[idx2[1]])
ratios = np.array(ratios); peak_freqs_all = np.array(peak_freqs_all)
print(f"\n(일반화) 길이 50 정상 세그먼트 {len(long_segs)}개 전부의 FFT 최댓값 성분 주파수: {np.unique(peak_freqs_all)} (전부 0.6Hz)")
print(f"2위 대비 진폭비: 중앙값 {np.median(ratios):.1f}배, 최소 {ratios.min():.1f}배, 최대 {ratios.max():.1f}배 → 예외 없이 단일 성분")

cands_coarse = sc.alias_candidates(peak_freq, fs=10.0, true_freq_range=(55, 65), step=0.1)
print("\n60Hz 부근 실제 주파수 후보(0.1Hz 해상도, FFT 빈 0.6Hz 기준):")
display(cands_coarse.head(4))
cands_fine = sc.alias_candidates(0.5687, fs=10.0, true_freq_range=(55, 65), step=0.01)
print("60Hz 부근 실제 주파수 후보(0.01Hz 해상도, zero-crossing 관측치 0.5687Hz 기준):")
display(cands_fine.head(4))
print("두 방식 모두 60Hz±0.57~0.6Hz(59.4/59.43 또는 60.57/60.6) 근방을 가리킨다. 이 차이는 (a) 실제 구동 주파수가 60Hz에서 벗어났거나, (b) 샘플링 클럭 자체가 이론값 10.000Hz에서 약 1% 벗어났거나(10.057Hz 등) 어느 쪽으로도 설명 가능하며, 이 데이터만으로는 두 가설을 구분할 수 없다(추정).")

대표 세그먼트(정상, 50샘플) FFT 상위 5개 성분:
  freq=0.6000 Hz  amp=57.52
  freq=1.8000 Hz  amp=3.32
  freq=3.0000 Hz  amp=1.67
  freq=4.2000 Hz  amp=1.41
  freq=0.4000 Hz  amp=0.18

최댓값 성분 freq=0.6000 Hz, 2위 대비 진폭비=17.3배 → 사실상 단일 성분



(일반화) 길이 50 정상 세그먼트 202개 전부의 FFT 최댓값 성분 주파수: [0.6] (전부 0.6Hz)
2위 대비 진폭비: 중앙값 22.6배, 최소 13.7배, 최대 56.9배 → 예외 없이 단일 성분

60Hz 부근 실제 주파수 후보(0.1Hz 해상도, FFT 빈 0.6Hz 기준):


,f_true_hz,n,alias_freq_hz,diff_from_observed
0,60.6,6,0.6,0.0
1,59.4,6,0.6,0.0
2,59.3,6,0.7,0.1
3,59.5,6,0.5,0.1


60Hz 부근 실제 주파수 후보(0.01Hz 해상도, zero-crossing 관측치 0.5687Hz 기준):


,f_true_hz,n,alias_freq_hz,diff_from_observed
0,59.43,6,0.57,0.0013
1,60.57,6,0.57,0.0013
2,60.56,6,0.56,0.0087
3,59.44,6,0.56,0.0087


두 방식 모두 60Hz±0.57~0.6Hz(59.4/59.43 또는 60.57/60.6) 근방을 가리킨다. 이 차이는 (a) 실제 구동 주파수가 60Hz에서 벗어났거나, (b) 샘플링 클럭 자체가 이론값 10.000Hz에서 약 1% 벗어났거나(10.057Hz 등) 어느 쪽으로도 설명 가능하며, 이 데이터만으로는 두 가설을 구분할 수 없다(추정).


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(g["ts"], g["AI2_Current"], marker=".", lw=.8)
axes[0].set_title(f"대표 정상 세그먼트(seg {rep_seg}, {len(g)}샘플) 파형")
axes[1].stem(freqs, amp, basefmt=" ")
axes[1].set_xlim(0, 5); axes[1].set_xlabel("Hz"); axes[1].set_ylabel("amplitude")
axes[1].set_title(f"FFT — 피크는 {peak_freq:.2f} Hz 근방 하나뿐 (정상 길이50 세그먼트 202개 전부 동일)")
save("02_fft_alias.png")

saved 02_fft_alias.png


In [16]:
ac_cur = sc.autocorr(g["AI2_Current"].to_numpy(), max_lag=20)
for ch in ["AI0_Vibration", "AI1_Vibration"]:
    ac_v = sc.autocorr(g[ch].to_numpy(), max_lag=20)
    zc_v = dq.zero_crossing_period(g[ch].to_numpy())
    print(f"{ch}: zero-crossing 주기(대표세그먼트)={zc_v:.2f}샘플, lag1 자기상관={ac_v[1]:.3f}")
print(f"AI2_Current: zero-crossing 주기(대표세그먼트)={dq.zero_crossing_period(g['AI2_Current'].to_numpy()):.2f}샘플, lag1 자기상관={ac_cur[1]:.3f}")

rows = []
for seg_id, gg in N.groupby("seg"):
    if len(gg) < 20:
        continue
    rows.append((dq.zero_crossing_period(gg["AI2_Current"].to_numpy()),
                 dq.zero_crossing_period(gg["AI0_Vibration"].to_numpy()),
                 dq.zero_crossing_period(gg["AI1_Vibration"].to_numpy())))
per_seg = pd.DataFrame(rows, columns=["pc", "p0", "p1"]).replace([np.inf, -np.inf], np.nan).dropna()
print(f"\n세그먼트별(n={len(per_seg)}) 전류-진동 zero-crossing 주기 상관: AI0 r={per_seg['pc'].corr(per_seg['p0']):.4f}, AI1 r={per_seg['pc'].corr(per_seg['p1']):.4f}")
print("상관이 0에 가까우면 진동 채널의 주기 구조가 전류(전원 유래 성분)와 무관 → 전원 노이즈 혼입 근거 약함")

AI0_Vibration: zero-crossing 주기(대표세그먼트)=5.26샘플, lag1 자기상관=0.338
AI1_Vibration: zero-crossing 주기(대표세그먼트)=4.00샘플, lag1 자기상관=-0.088
AI2_Current: zero-crossing 주기(대표세그먼트)=16.67샘플, lag1 자기상관=0.911

세그먼트별(n=452) 전류-진동 zero-crossing 주기 상관: AI0 r=-0.0136, AI1 r=0.0036
상관이 0에 가까우면 진동 채널의 주기 구조가 전류(전원 유래 성분)와 무관 → 전원 노이즈 혼입 근거 약함


**3절 결과**: FFT·주파수 도메인 분석이 "무의미"하다는 01 리포트의 주장이 직접 검증됐고(길이 50 정상 202세그먼트 전부 0.6Hz 단일 피크, 2위 대비 중앙값 22.6배·최소 13.7배), 60Hz대 후보는 59.4/59.43~60.57/60.6Hz 부근이며 정확한 원인(구동주파수 편차 vs 샘플링 클럭 오차)은 이 데이터로 확정할 수 없다(추정). **이상 구간 일부 세그먼트(16~20)에서 zero-crossing이 사라지는 현상은 "AC 성분이 흐트러져서"가 아니라 2절에서 확인한 큰 DC 편이가 부호 변화를 가렸기 때문이며, 세그먼트 평균을 빼면 교차가 다시 나타난다(위 표 확인).** 진동 채널은 전류와 무관한 자체적(광대역) 신호로 보이며 별도 노치 필터가 필요하지 않다.

**모델 단계 반영**: FFT·주파수 도메인 피처는 **사용하지 않는다**. 시간영역 진폭 피처(RMS·피크·첨도·crest factor)만 사용한다. zero-crossing이 0인 세그먼트를 "AC 신호 소실"로 오독하지 말고, 반드시 DC 제거 후 재계산해서 판단한다(2절의 오프셋 제거 전처리와 연결).

## 4. 이상 구간의 시간 진행 (B-2 ④)

- **가설**: 이상 21세그먼트를 시간순으로 보면 점진적으로 악화되는 추세가 있을 수 있다(조기탐지 스토리). 600행이 정확히 12×50(=12개 완전한 burst)인지, 첫·끝 세그먼트가 잘렸는지 확인한다. **(v2 추가)** 진동 지표(vib_rms)뿐 아니라 전류 DC·RMS도 시간 추세를 봐야 한다 — 2·3절에서 전류 DC가 후반 세그먼트(16~20)에 몰려 있었으므로 시간에 따라 증가할 가능성이 있다.
- **실험**: `segments.outlier_trend`로 시간순 `vib_rms`, `AI2_Current_dc`(부호 있는 값), `AI2_Current_rms`의 선형 기울기·Spearman 상관을 각각 구했다.

In [17]:
trend_vib = sg.outlier_trend(fO, col="vib_rms")
trend_dc = sg.outlier_trend(fO, col="AI2_Current_dc")
trend_rms = sg.outlier_trend(fO, col="AI2_Current_rms")
print("vib_rms 추세:", trend_vib)
print("AI2_Current_dc(부호 있음) 추세:", trend_dc)
print("AI2_Current_rms 추세:", trend_rms)
print(f"\n세그먼트 길이 합계: {fO['len'].sum()} (12×50={12*50}과 비교) — 21개 세그먼트, 길이 최소 {fO['len'].min()}~최대 {fO['len'].max()}")
print(f"첫 세그먼트 길이 {fO.sort_values('start')['len'].iloc[0]}, 마지막 세그먼트 길이 {fO.sort_values('start')['len'].iloc[-1]} — 둘 다 50 미만이면 절단 가능성")

vib_rms 추세: {'slope': -0.01842623966811663, 'intercept': 0.6940607405591279, 'spearman_rho': -0.42987012987012985, 'spearman_p': 0.05178188171189559}
AI2_Current_dc(부호 있음) 추세: {'slope': 14.916830243815497, 'intercept': -113.6488021731444, 'spearman_rho': 0.6597402597402597, 'spearman_p': 0.0011382402645393446}
AI2_Current_rms 추세: {'slope': 6.598775902925473, 'intercept': 80.54391686855273, 'spearman_rho': 0.5779220779220778, 'spearman_p': 0.006071212564747679}

세그먼트 길이 합계: 600 (12×50=600과 비교) — 21개 세그먼트, 길이 최소 3~최대 50
첫 세그먼트 길이 4, 마지막 세그먼트 길이 15 — 둘 다 50 미만이면 절단 가능성


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fo_sorted = fO.sort_values("start").reset_index()
t = np.arange(len(fo_sorted))
axes[0].plot(t, fo_sorted["vib_rms"], marker="o", color="steelblue")
z = np.polyfit(t, fo_sorted["vib_rms"], 1)
vib_label = f"기울기={trend_vib['slope']:.4f} (p={trend_vib['spearman_p']:.3f})"
axes[0].plot(t, np.polyval(z, t), "--", c="steelblue", label=vib_label)
axes[0].set_xlabel("세그먼트 순번(시간순)"); axes[0].set_ylabel("vib_rms"); axes[0].set_title("진동: 약한 감소 경향(유의성 경계)")
axes[0].legend(fontsize=8)

axes[1].plot(t, fo_sorted["AI2_Current_dc"], marker="o", color="crimson", label="AI2_Current_dc")
axes[1].plot(t, fo_sorted["AI2_Current_rms"], marker="s", color="darkorange", label="AI2_Current_rms")
dc_label = f"DC rho={trend_dc['spearman_rho']:.2f}(p={trend_dc['spearman_p']:.3f}), RMS rho={trend_rms['spearman_rho']:.2f}(p={trend_rms['spearman_p']:.3f})"
axes[1].set_title(dc_label, fontsize=9)
axes[1].set_xlabel("세그먼트 순번(시간순)"); axes[1].legend(fontsize=8)
save("02_outlier_trend.png")

saved 02_outlier_trend.png


**4절 결과(v2, 결론 수정)**: 진동(`vib_rms`)은 시간이 지날수록 약하게 감소하는 경향(기울기 -0.0184/세그먼트, Spearman rho=-0.430, **p=0.052**로 유의수준 경계)을 보였다. 반면 **전류는 뚜렷이 유의하게 증가한다** — `AI2_Current_dc`(부호 있는 값) rho=**0.66, p=0.001**, `AI2_Current_rms` rho=**0.58, p=0.006**. 즉 "이상 이벤트가 시간에 따라 완화된다"는 1차 결론은 진동 채널에만 국한된 약한(통계적으로 확정 못한) 관찰이었고, **전류 DC/RMS 기준으로는 오히려 시간이 지날수록 뚜렷이 악화(또는 오프셋이 누적)되는 추세**가 확인됐다. 600행이 정확히 12×50과 같은 것은 길이가 3~50으로 흩어진 21개 세그먼트 합의 우연이며, 첫(4)·끝(15) 세그먼트 길이가 50 미만이라 관측 구간이 잘렸을 가능성이 있다(추정).

**모델 단계 반영**: "점진 악화형 조기탐지" 스토리는 채널마다 다른 결론을 준다 — 진동만 보면 근거가 약하지만, 전류(특히 DC/오프셋)를 포함하면 유의한 시간 추세가 있다. 다만 이 전류 추세가 2절의 "파일 형식/오프셋 누수"와 얽혀 있을 수 있어(오프셋이 시간에 따라 커지는 것이 실제 설비 열화인지 계측 드리프트인지 이 데이터로는 확정 불가), 시간 추세 자체를 피처로 쓰기보다는 **리포트에 한계로 남기고 조기탐지 주장에 신중을 기한다.**

## 5. burst 수집 규칙 (B-2 ⑤)

- **가설**: burst는 주기적 폴링(일정 간격)으로 수집되며, 이 간격이 탐지 지연의 물리적 하한을 결정한다.
- **실험**: 세그먼트 시작 시각 간격 분포, 총 가동시간 대비 duty cycle.

In [19]:
starts_n = N.groupby("seg")["ts"].min().sort_values()
iv_n = starts_n.diff().dt.total_seconds().dropna()
print("normal burst 시작 간격(초):"); display(iv_n.describe())
print(f"중앙값={iv_n.median():.3f}s, IQR=[{iv_n.quantile(.25):.3f}, {iv_n.quantile(.75):.3f}]")

starts_o = O.groupby("seg")["ts"].min().sort_values()
iv_o = starts_o.diff().dt.total_seconds().dropna()
print("\noutlier burst 시작 간격(초):"); display(iv_o.describe())

dur_sum = dq.segment_table(N)["dur_s"].sum()
elapsed = (N["ts"].max() - N["ts"].min()).total_seconds()
print(f"\nnormal 총 가동(burst 내부) 시간 {dur_sum:.1f}s / 전체 경과 {elapsed:.1f}s → duty cycle {dur_sum/elapsed:.4f}")
print(f"burst 간격 중앙값 {iv_n.median():.2f}s ≈ 다음 burst가 와야 값을 볼 수 있는 시간 → 규칙 기반 탐지 지연의 물리적 하한")

gap01 = (starts_o.loc[1] - starts_o.loc[0]).total_seconds()
print(f"\n(8절에서 사용) 이상 이벤트의 첫 burst(세그먼트 0, 길이 4, 판정 불가)에서 다음 burst(세그먼트 1)까지 간격: {gap01:.3f}s")

normal burst 시작 간격(초):


count    598.000000
mean       7.711554
std        1.698796
min        1.545000
25%        7.299000
50%        7.958000
75%        8.057750
max       17.643000
Name: ts, dtype: float64

중앙값=7.958s, IQR=[7.299, 8.058]

outlier burst 시작 간격(초):


count    20.000000
mean      8.209850
std       0.732269
min       6.680000
25%       8.071750
50%       8.129500
75%       8.486000
max       9.855000
Name: ts, dtype: float64


normal 총 가동(burst 내부) 시간 1940.0s / 전체 경과 4615.8s → duty cycle 0.4203
burst 간격 중앙값 7.96s ≈ 다음 burst가 와야 값을 볼 수 있는 시간 → 규칙 기반 탐지 지연의 물리적 하한

(8절에서 사용) 이상 이벤트의 첫 burst(세그먼트 0, 길이 4, 판정 불가)에서 다음 burst(세그먼트 1)까지 간격: 8.781s


**5절 결과**: (수치는 위 출력에서 인용)

**모델 단계 반영**: 현장 활용안의 "탐지 지연" 수치를 보고할 때 burst 간격 중앙값을 **구조적 하한**으로 같이 명시한다(신호 처리로는 더 줄일 수 없음). duty cycle이 낮다는 것은 실제 신호 취득이 간헐적이라는 뜻이므로, 실시간 연속 모니터링이 아니라 "주기적 스냅샷 점검" 프레임으로 현장 활용안을 서술한다.

## 6. 채널 간 관계 (B-2 ⑥)

- **가설**: 정상 구간에서는 부하(전류)와 진동이 비례하므로, 부하로 보정한 진동 지표(진동 RMS/전류 RMS)가 원시 진동 RMS보다 정상/이상 분리에 유리하거나, 적어도 부하 변화로 인한 오경보를 줄일 수 있다.
- **실험**: 세그먼트별 AI0-AI1 원시신호 상관, 전류 RMS-진동 RMS 산점(정상/이상), `vib_rms` 단독 AUC vs 부하보정지표(`vib_rms/AI2_Current_rms`) 단독 AUC 비교.

In [20]:
corr01 = sg.within_segment_corr(N, "AI0_Vibration", "AI1_Vibration", min_len=5)
print(f"정상 세그먼트별 AI0-AI1 상관(n={len(corr01)}): mean={corr01.mean():.4f}, median={corr01.median():.4f}")
print(f"normal 전류RMS-진동RMS(vib_rms) 상관: {fN['AI2_Current_rms'].corr(fN['vib_rms']):.4f}")

fN["load_corr"] = fN["vib_rms"] / fN["AI2_Current_rms"]
fO["load_corr"] = fO["vib_rms"] / fO["AI2_Current_rms"]
auc_vib = sg.separability_auc(fN, fO, "vib_rms")
auc_loadcorr = sg.separability_auc(fN, fO, "load_corr")
print(f"\nvib_rms 단독 AUC = {auc_vib:.4f}")
print(f"load_corr(vib_rms/current_rms) 단독 AUC = {auc_loadcorr:.4f}")
print("부하보정 지표가 원시 vib_rms보다 AUC가 낮으면, 이 데이터에서는 부하 보정이 분리력을 개선하지 못한다는 뜻 (추정: 이상 구간의 진동 증가폭이 워낙 커서 원시값만으로도 이미 포화 수준으로 분리됨)")

정상 세그먼트별 AI0-AI1 상관(n=570): mean=0.2334, median=0.2803
normal 전류RMS-진동RMS(vib_rms) 상관: 0.8913

vib_rms 단독 AUC = 0.8946
load_corr(vib_rms/current_rms) 단독 AUC = 0.8629
부하보정 지표가 원시 vib_rms보다 AUC가 낮으면, 이 데이터에서는 부하 보정이 분리력을 개선하지 못한다는 뜻 (추정: 이상 구간의 진동 증가폭이 워낙 커서 원시값만으로도 이미 포화 수준으로 분리됨)


In [21]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(fN["AI2_Current_rms"], fN["vib_rms"], s=10, alpha=.4, label="normal", c="steelblue")
ax.scatter(fO["AI2_Current_rms"], fO["vib_rms"], s=25, alpha=.8, label="outlier", c="crimson", marker="x")
ax.set_xlabel("AI2_Current_rms"); ax.set_ylabel("vib_rms"); ax.set_title("세그먼트별 전류 RMS - 진동 RMS 관계")
ax.legend()
save("02_channel_relation.png")

saved 02_channel_relation.png


**6절 결과**: (수치는 위 출력에서 인용)

**모델 단계 반영**: `load_corr`가 `vib_rms`보다 낫지 않다면 주 피처는 원시 `vib_rms`(및 AI0/AI1 개별 RMS)로 두고, `load_corr`는 "부하 변화에 따른 오경보 억제"가 필요한 운전 상태(예: **1절**의 저부하 state)에 한정된 **보조 규칙**으로만 쓴다. AI0-AI1 상관이 중간 정도(0.2~0.4대)이면 두 채널을 평균하지 않고 **개별 피처로 유지**한다(서로 다른 정보가 있다는 뜻).

## 7. 윈도우 길이·정상량 민감도 (B-2 ⑦)

- **가설**: 윈도우가 길수록(1→5초) 진동 RMS의 정상/이상 분리력(AUC)이 좋아지지만, 전류 RMS는 에일리어싱 때문에 윈도우를 늘려도 크게 개선되지 않는다. 또한 임계값을 정상 데이터 앞부분 일부(20~80%)만으로 잡아도 오경보율이 크게 늘지 않으면, 적은 정상 데이터로도 규칙을 세울 수 있다.
- **(v2 주의, 선택 효과)**: 윈도우가 길어질수록 그 길이를 채울 수 있는 이상 세그먼트 수 자체가 줄어든다(5초=50샘플 윈도우는 길이 50인 이상 세그먼트 **5개**만 참여). 전체 세그먼트를 쓴 표와, 세그먼트 집합을 길이 50으로 고정한 공정 비교표를 함께 낸다.
- **실험**: `segments.window_auc_table`(전체 세그먼트, `n_outlier_segments_qualify` 포함), `segments.window_auc_fixed_length`(길이 50 고정 집합), `segments.leakage_free_fpr_table`(시간순 앞 20/40/60/80% 세그먼트로 99분위 임계값을 잡고 나머지에서 **샘플 단위·세그먼트 단위** 오경보율 둘 다).

In [22]:
wt = sg.window_auc_table(N, O, windows_sec=(1, 2, 3, 5))
print("전체 세그먼트 사용 (윈도우가 길수록 참여 가능한 이상 세그먼트 수가 줄어드는 선택 효과 있음):")
display(wt.pivot(index="window_sec", columns="channel", values="auc"))
display(wt.pivot(index="window_sec", columns="channel", values="n_outlier_segments_qualify"))

전체 세그먼트 사용 (윈도우가 길수록 참여 가능한 이상 세그먼트 수가 줄어드는 선택 효과 있음):


channel,AI0_Vibration,AI1_Vibration,AI2_Current
window_sec,,,
1,0.8067,0.7913,0.5194
2,0.8771,0.8509,0.5023
3,0.9182,0.8876,0.5289
5,1.0000,0.9782,0.6347


channel,AI0_Vibration,AI1_Vibration,AI2_Current
window_sec,,,
1,17,17,17
2,13,13,13
3,10,10,10
5,5,5,5


In [23]:
wtf = sg.window_auc_fixed_length(N, O, length=50, windows_sec=(1, 2, 3, 5))
print(f"길이 50으로 고정한 세그먼트만 사용 (정상 {wtf.n_normal_segments_fixed.iloc[0]}개, 이상 {wtf.n_outlier_segments_fixed.iloc[0]}개 — 4개 윈도우 크기 전부 동일 집합):")
display(wtf.pivot(index="window_sec", columns="channel", values="auc"))
print("동일 집합에서는 1→3초로 갈수록 AUC가 개선되는 추세가 유지된다(AI0 0.76→0.89). 5초 지점은 이상 세그먼트가 애초에 5개뿐이라(길이 50 제약) 신뢰도 낮은 값으로 별도 취급한다.")

길이 50으로 고정한 세그먼트만 사용 (정상 202개, 이상 5개 — 4개 윈도우 크기 전부 동일 집합):


channel,AI0_Vibration,AI1_Vibration,AI2_Current
window_sec,,,
1,0.7588,0.7648,0.5348
2,0.8398,0.8061,0.5074
3,0.8900,0.8490,0.5475
5,1.0000,0.9782,0.6347


동일 집합에서는 1→3초로 갈수록 AUC가 개선되는 추세가 유지된다(AI0 0.76→0.89). 5초 지점은 이상 세그먼트가 애초에 5개뿐이라(길이 50 제약) 신뢰도 낮은 값으로 별도 취급한다.


In [24]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ch in dq.SENSORS:
    sub = wt[wt.channel == ch]
    axes[0].plot(sub["window_sec"], sub["auc"], marker="o", label=ch)
axes[0].axhline(0.5, ls=":", c="gray")
axes[0].set_xlabel("윈도우 길이(초)"); axes[0].set_ylabel("AUC"); axes[0].set_title("전체 세그먼트 (선택 효과 있음)")
axes[0].legend(fontsize=8)
for ch in dq.SENSORS:
    sub = wtf[wtf.channel == ch]
    axes[1].plot(sub["window_sec"], sub["auc"], marker="s", label=ch)
axes[1].axhline(0.5, ls=":", c="gray")
axes[1].set_xlabel("윈도우 길이(초)"); axes[1].set_ylabel("AUC"); axes[1].set_title("길이 50 고정 집합 (공정 비교)")
axes[1].legend(fontsize=8)
save("02_window_auc.png")

saved 02_window_auc.png


In [25]:
fpr_tab = sg.leakage_free_fpr_table(N, win=10, channel="AI0_Vibration", fractions=(0.2, 0.4, 0.6, 0.8))
display(fpr_tab)
print("샘플 단위 오경보율(fpr_sample)은 1~1.5% 수준으로 낮아 보이지만, 세그먼트(=한 번의 점검 burst) 단위 오경보율(fpr_segment)은 이보다 훨씬 높다(20% 학습 시 7.67%) — 현장 운영에서 실제로 체감하는 오경보 빈도는 세그먼트 단위에 더 가깝다.")

,train_frac,n_train_seg,n_test_seg,threshold_q99,n_test_samples,fpr_sample,n_test_seg_valid,fpr_segment
0,0.2,120,479,0.1251,11521,0.0147,417,0.0767
1,0.4,240,359,0.1270,8745,0.0134,315,0.0635
2,0.6,359,240,0.1277,5964,0.0132,217,0.0599
3,0.8,479,120,0.1320,3043,0.0023,109,0.0092


샘플 단위 오경보율(fpr_sample)은 1~1.5% 수준으로 낮아 보이지만, 세그먼트(=한 번의 점검 burst) 단위 오경보율(fpr_segment)은 이보다 훨씬 높다(20% 학습 시 7.67%) — 현장 운영에서 실제로 체감하는 오경보 빈도는 세그먼트 단위에 더 가깝다.


**7절 결과(v2, 결론 수정)**: 전체 세그먼트를 쓰면 5초 윈도우에서 AI0 AUC가 1.0000까지 오르지만, 이는 **길이 50인 이상 세그먼트 5개만 남긴 선택 효과**이므로 "5초가 최고"라고 결론짓지 않는다. 세그먼트 집합을 고정(길이 50, 정상 202개·이상 5개)해 공정 비교하면 **1→3초 구간에서 AUC가 꾸준히 개선**되고(AI0 0.7588→0.8900), 5초 지점은 표본이 5개뿐이라 비교 대상에서 제외한다. AI1은 3초에서 AUC 0.8876~0.8900 수준으로 "3초 이상이면 0.9를 넘는다"고 말하기는 이르다. 오경보율은 **샘플 단위**(1.47%→0.23%, 정상 20%→80% 학습)보다 **세그먼트 단위**(7.67%→0.92%)가 훨씬 높게 나와, 운영 관점의 실제 오경보 빈도는 세그먼트 단위로 봐야 한다.

**모델 단계 반영**: 윈도우 길이는 **1~3초** 사이에서 고른다("5초가 최적"이라는 이전 결론은 폐기). 오경보율은 반드시 **샘플 단위와 세그먼트 단위를 함께** 보고한다(세그먼트 단위가 운영 관점에 더 가까움). 정상 데이터 20%만으로 잡은 임계값의 세그먼트 단위 오경보율(7.67%)을 콜드스타트 시나리오의 기준 수치로 삼는다.

## 8. 규칙 기반 탐지 지연 (B-2 ⑧)

- **가설**: 정상 전체의 99분위 임계값을 넘는 첫 샘플까지 걸리는 시간(지연)은 대부분 1초 안팎이고, 조용한 세그먼트(3·19·20)는 임계값을 넘지 못해 규칙 기반으로는 미탐지된다. **(v2 추가)** 지연 9샘플은 1초(10샘플) 윈도우가 처음 완성되는 시점(=최솟값)일 수 있고, 세그먼트 0(길이 4, 판정 불가)을 감안하면 이벤트 시작부터의 실제 지연은 burst 간격만큼 더 길다.
- **실험**: `segments.detection_delay_table`로 1초 윈도우(AI0_Vibration) 기준 세그먼트별 최초 초과 위치를 구한다. 세그먼트 길이가 윈도우보다 짧아 판정 자체가 불가능한 경우와, 조용해서 미탐지인 경우를 구분한다.

In [26]:
thr_full = dq.rolling_rms(N, win=10)["AI0_Vibration"].dropna().quantile(0.99)
print(f"정상 전체(1초 윈도우, AI0_Vibration) q99 임계값 = {thr_full:.4f}")
delay_tab = sg.detection_delay_table(O, threshold=thr_full, win=10, channel="AI0_Vibration")
display(delay_tab)
print(delay_tab["reason"].value_counts())
det = delay_tab[delay_tab.detected]
print(f"\n탐지된 세그먼트 {len(det)}/{len(delay_tab)}, 지연(샘플) 중앙값={det['delay_samples'].median():.1f} (={det['delay_samples'].median()/10:.2f}s), 평균={det['delay_samples'].mean():.2f}")
print(f"지연=9샘플은 1초(10샘플) 윈도우가 세그먼트 시작 후 '처음 완성되는' 시점(=win-1)과 같다 — 즉 많은 세그먼트가 첫 윈도우 완성 즉시 탐지되는, 사실상 가능한 최솟값이다.")
gap01 = (O.groupby("seg")["ts"].min().sort_index().loc[1] - O.groupby("seg")["ts"].min().sort_index().loc[0]).total_seconds()
delay_med_s = det["delay_samples"].median()/10
print(f"이벤트 실제 시작(세그먼트 0, 길이 4, 판정 불가)부터 헤아리면: burst 간격 {gap01:.2f}s(세그먼트0→1) + 윈도우 내 지연 {delay_med_s:.2f}s ≈ 실제 체감 지연 {gap01+delay_med_s:.2f}s")

정상 전체(1초 윈도우, AI0_Vibration) q99 임계값 = 0.1296


,seg,n,n_valid_windows,detected,delay_samples,delay_sec,reason
0,0,4,0,False,NaN,NaN,세그먼트 길이<윈도우(판정 불가)
1,1,50,41,True,9.0,0.9,탐지
2,2,47,38,True,13.0,1.3,탐지
3,3,31,22,False,NaN,NaN,조용한 세그먼트(라벨 노이즈 의심)
4,4,18,9,True,9.0,0.9,탐지
5,5,3,0,False,NaN,NaN,세그먼트 길이<윈도우(판정 불가)
6,6,50,41,True,9.0,0.9,탐지
7,7,40,31,True,9.0,0.9,탐지
8,8,25,16,True,17.0,1.7,탐지
9,9,8,0,False,NaN,NaN,세그먼트 길이<윈도우(판정 불가)


reason
탐지                     14
세그먼트 길이<윈도우(판정 불가)      4
조용한 세그먼트(라벨 노이즈 의심)     3
Name: count, dtype: int64

탐지된 세그먼트 14/21, 지연(샘플) 중앙값=9.0 (=0.90s), 평균=11.64
지연=9샘플은 1초(10샘플) 윈도우가 세그먼트 시작 후 '처음 완성되는' 시점(=win-1)과 같다 — 즉 많은 세그먼트가 첫 윈도우 완성 즉시 탐지되는, 사실상 가능한 최솟값이다.
이벤트 실제 시작(세그먼트 0, 길이 4, 판정 불가)부터 헤아리면: burst 간격 8.78s(세그먼트0→1) + 윈도우 내 지연 0.90s ≈ 실제 체감 지연 9.68s


In [27]:
fig, ax = plt.subplots(figsize=(8, 4))
reason_colors = {"탐지": "steelblue", "조용한 세그먼트(라벨 노이즈 의심)": "orange", "세그먼트 길이<윈도우(판정 불가)": "gray", "미탐지": "crimson"}
for reason, sub in delay_tab.groupby("reason"):
    y = sub["delay_samples"].fillna(-2)
    ax.scatter(sub["seg"], y, label=reason, c=reason_colors.get(reason, "black"), s=40)
ax.set_xlabel("이상 세그먼트 번호"); ax.set_ylabel("탐지 지연(샘플, 1초윈도우=10샘플/초)")
ax.set_title("세그먼트별 규칙 기반 탐지 지연")
ax.legend(fontsize=8, loc="upper left")
save("02_detection_delay.png")

saved 02_detection_delay.png


**8절 결과**: 미탐지 세그먼트 중 3·19·20은 진동이 실제로 조용해 라벨 노이즈로 의심되는 것과, 나머지는 세그먼트 길이가 1초 윈도우(10샘플)보다 짧아 애초에 판정 불가능한 경우로 나뉜다. 지연 중앙값 9샘플(=0.9초, 1초 윈도우 기준)은 **윈도우가 처음 완성되는 시점(=win-1)과 같은 사실상의 최솟값**이라, "매우 빠르게 탐지된다"는 해석은 신중해야 한다. 이벤트가 실제로 시작한 시점(판정 불가한 세그먼트 0)부터 헤아리면, 다음 burst까지의 구조적 지연(8.78초)을 더해 실제 체감 지연은 약 9.68초에 가깝다.

**모델 단계 반영**: 지표 정의를 "세그먼트당 탐지 지연(중앙값 0.9초, **1초 윈도우 기준**)"과 "판정 불가 세그먼트 비율(길이<윈도우)"으로 분리해서 보고하되, 현장 활용안에는 **burst 간격을 더한 실제 체감 지연(≈9.68초)**을 반드시 함께 제시한다.

## 9. 모델 단계 반영 사항 요약

| 항목 | 규칙 |
|---|---|
| split 키 | 세그먼트(`seg_uid`) 단위 GroupKFold. 정상은 추가로 시간순 블록(앞/뒤) 분할 병행 |
| soft label | 채널별로 분리(`vib_grade`, `cur_grade`). 세그먼트 19·20처럼 진동=정상 유사/전류=확실 이상인 경우를 "정상"으로 단정하지 않는다 |
| 전처리(오프셋) | **세그먼트 평균(DC 오프셋) 제거를 필수 전처리로 채택**(|DC| AUC 0.89~0.94로 매우 높음 — v2에서 반전). 오프셋 제거 후에도 `|DC|` 자체는 별도 피처로 남긴다 |
| 전처리(형식 통일) | 전류 양자화 배수(99.67% vs 0.19%)·진동 소수 자릿수(6 vs 8자리)가 파일마다 달라 원시값 반올림 등으로 형식을 통일한다 |
| 윈도우 길이 | 1~3초(길이 고정 비교에서 AUC 개선 확인). "5초가 최적"이라는 결론은 선택 효과로 폐기 |
| 주 피처 | AI0/AI1 진동 RMS·피크·첨도·crest factor (시간영역만, FFT/스펙트럼 피처 금지 — 3절 근거) |
| 보조 피처 | 전류 RMS·피크(AUC 약함, 보조), `load_corr`는 1절의 저부하 상태 한정 보조 규칙 |
| 지표 정의 | 세그먼트 단위 탐지율(14/21), 오경보율은 **샘플 단위·세그먼트 단위 모두** 보고(세그먼트 단위가 운영 관점에 더 가까움), 탐지 지연(중앙값 0.9초, **1초 윈도우 기준**, 이벤트 시작 대비 실제 체감 지연은 burst 간격 포함 ≈9.68초), 판정 불가 세그먼트 비율(19%, 길이<윈도우) |

리포트 `reports/02_diagnosis_deep.md`에 위 표와 세부 수치를 정리한다.